# 07.11 Split and Join — Deep Dive

Six methods break strings apart and put them back together. This family has the
most parameters and the most surprising edge cases of any group.

| Method | Signature | Returns |
|---|---|---|
| `split` | `split(sep=None, maxsplit=-1)` | list of parts |
| `rsplit` | `rsplit(sep=None, maxsplit=-1)` | list, splitting from the right |
| `splitlines` | `splitlines(keepends=False)` | list of lines |
| `join` | `join(iterable)` | one string |
| `partition` | `partition(sep)` | **always** a 3-tuple |
| `rpartition` | `rpartition(sep)` | 3-tuple, searching from the right |

The single most important detail: **`split()` and `split(" ")` behave
differently**. That difference causes real bugs.

## Easy — Breaking text apart and rebuilding it

The basic operations, one at a time.

In [ ]:
# EXAMPLE 1: Splitting a sentence into words
sentence = "the quick brown fox"

print(sentence.split())

In [ ]:
# EXAMPLE 2: The result is a list
sentence = "the quick brown fox"

words = sentence.split()

print("type:", type(words).__name__)
print("length:", len(words))
print("first word:", words[0])

In [ ]:
# EXAMPLE 3: Splitting on a comma
csv_line = "name,age,city"

print(csv_line.split(","))

In [ ]:
# EXAMPLE 4: Splitting on any string, not just one character
text = "one -- two -- three"

print(text.split(" -- "))

In [ ]:
# EXAMPLE 5: Joining a list back together
# join() is called ON the separator. This surprises everyone at first.
words = ["the", "quick", "fox"]

print(" ".join(words))

In [ ]:
# EXAMPLE 6: Joining with different separators
words = ["a", "b", "c"]

print(" ".join(words))
print(",".join(words))
print(" -> ".join(words))
print("".join(words))

In [ ]:
# EXAMPLE 7: Split and join are opposites
original = "the quick brown fox"

words = original.split()
rebuilt = " ".join(words)

print("original:", original)
print("rebuilt: ", rebuilt)
print("Equal?", original == rebuilt)

In [ ]:
# EXAMPLE 8: Splitting lines of text
document = "first line\nsecond line\nthird line"

for line in document.splitlines():
    print(line)

In [ ]:
# EXAMPLE 9: join() needs strings, not numbers
numbers = [1, 2, 3]

try:
    ",".join(numbers)
except TypeError as error:
    print("TypeError:", error)

print("Fixed:", ",".join(str(number) for number in numbers))

## Medium — The parameters and the big gotcha

maxsplit, keepends, and the split() vs split(' ') difference.

In [ ]:
# EXAMPLE 10: THE BIG ONE: split() versus split(' ')
# This is the most important cell in the notebook.
messy = "  a   b  "

print("split():    ", messy.split())
print("split(' '): ", messy.split(" "))

In [ ]:
# EXAMPLE 11: Why they differ
messy = "  a   b  "

print("split() with NO argument:")
print("   - treats any run of whitespace as ONE separator")
print("   - ignores leading and trailing whitespace")
print("   - result:", messy.split())
print("")
print("split(' ') with an explicit space:")
print("   - splits on EVERY single space")
print("   - keeps the empty strings between them")
print("   - result:", messy.split(" "))

In [ ]:
# EXAMPLE 12: Which one you usually want
# For human-typed input, split() with no argument is almost always right.
user_input = "  Asha   Nikumbh  "

print("split():   ", user_input.split())
print("split(' '):", user_input.split(" "))
print("")
print("Use split() for words. Use split(sep) for structured data like CSV.")

In [ ]:
# EXAMPLE 13: split() also handles tabs and newlines
# Any whitespace counts, not just spaces.
mixed = "a\tb\nc  d"

print(repr(mixed))
print(mixed.split())

In [ ]:
# EXAMPLE 14: maxsplit limits how many splits happen
line = "key=value=extra=more"

print("no limit: ", line.split("="))
print("maxsplit=1:", line.split("=", 1))
print("maxsplit=2:", line.split("=", 2))

In [ ]:
# EXAMPLE 15: Why maxsplit matters for config parsing
# The value itself might contain the separator.
line = "message=hello=world"

# Wrong - the value gets broken up.
print("without maxsplit:", line.split("="))

# Right - split once, keep the rest intact.
key, value = line.split("=", 1)
print("key:  ", key)
print("value:", value)

In [ ]:
# EXAMPLE 16: rsplit() splits from the right
path = "folder/subfolder/file.txt"

print("split with maxsplit=1: ", path.split("/", 1))
print("rsplit with maxsplit=1:", path.rsplit("/", 1))

In [ ]:
# EXAMPLE 17: Using rsplit for filenames
# The classic use - separate the last component.
path = "a/b/c/document.txt"

directory, filename = path.rsplit("/", 1)

print("directory:", directory)
print("filename: ", filename)

In [ ]:
# EXAMPLE 18: rsplit for file extensions
filename = "archive.tar.gz"

print("split('.', 1): ", filename.split(".", 1))
print("rsplit('.', 1):", filename.rsplit(".", 1))
print("")
print("rsplit gets the final extension, which is usually what you want.")

In [ ]:
# EXAMPLE 19: splitlines() handles every newline convention
# Unix, Windows and old Mac line endings all work.
unix = "a\nb"
windows = "a\r\nb"
old_mac = "a\rb"

print("unix:   ", unix.splitlines())
print("windows:", windows.splitlines())
print("old mac:", old_mac.splitlines())

In [ ]:
# EXAMPLE 20: splitlines() versus split('\n')
text = "a\r\nb\r\nc"

print("splitlines():", text.splitlines())
print("split('\\n'):  ", text.split("\n"))
print("")
print("split leaves the stray carriage returns behind.")

In [ ]:
# EXAMPLE 21: keepends=True keeps the line endings
document = "first\nsecond\nthird"

print("keepends=False:", document.splitlines())
print("keepends=True: ", document.splitlines(keepends=True))

In [ ]:
# EXAMPLE 22: partition() always returns three parts
setting = "timeout=30"

before, separator, after = setting.partition("=")

print("before:   ", repr(before))
print("separator:", repr(separator))
print("after:    ", repr(after))

In [ ]:
# EXAMPLE 23: partition() when the separator is missing
# Unlike split, you always get exactly three items.
setting = "no separator here"

before, separator, after = setting.partition("=")

print("before:   ", repr(before))
print("separator:", repr(separator))
print("after:    ", repr(after))
print("")
print("The separator being empty tells you it was not found.")

In [ ]:
# EXAMPLE 24: partition() versus split() for safety
lines = ["timeout=30", "malformed line"]

for line in lines:
    # partition never raises, and never returns the wrong number of parts.
    key, found, value = line.partition("=")

    if found:
        print(f"{line!r:<20} -> key={key!r} value={value!r}")
    else:
        print(f"{line!r:<20} -> no separator, skipping")

In [ ]:
# EXAMPLE 25: rpartition() searches from the right
path = "a/b/c.txt"

print("partition: ", path.partition("/"))
print("rpartition:", path.rpartition("/"))

## Hard — Edge cases, performance and real parsing

The behaviour that catches people out in production code.

In [ ]:
# EXAMPLE 26: Splitting an empty string
# The two forms differ here too.
empty = ""

print("''.split():    ", empty.split())
print("''.split(','): ", empty.split(","))
print("")
print("With no separator you get []. With one you get [''].")

In [ ]:
# EXAMPLE 27: Splitting on an empty separator is an error
text = "abc"

try:
    text.split("")
except ValueError as error:
    print("ValueError:", error)

print("")
print("To split into characters, use list():", list(text))

In [ ]:
# EXAMPLE 28: Trailing separators create empty strings
line = "a,b,c,"

print(line.split(","))
print("")
print("The trailing comma produced a final empty string.")
print("Filter them out if that is not what you want:")
print([part for part in line.split(",") if part])

In [ ]:
# EXAMPLE 29: Consecutive separators create empty strings
line = "a,,b"

print(line.split(","))
print("")
print("This is correct for CSV - it means an empty field.")

In [ ]:
# EXAMPLE 30: Splitting preserves whitespace inside fields
csv_line = "name , age , city"

print("raw split:   ", csv_line.split(","))
print("with strip:  ", [field.strip() for field in csv_line.split(",")])

In [ ]:
# EXAMPLE 31: join() accepts any iterable, not just lists
# Sets, tuples, generators and dict keys all work.
print("from a tuple: ", "-".join(("a", "b", "c")))
print("from a set:   ", "-".join({"a"}))
print("from a range: ", "-".join(str(n) for n in range(3)))
print("from dict keys:", "-".join({"x": 1, "y": 2}))

In [ ]:
# EXAMPLE 32: join() on a string joins its characters
# A string is itself an iterable of characters.
print("-".join("abc"))
print("")
print("Occasionally useful, often a bug when you meant to pass a list.")

In [ ]:
# EXAMPLE 33: join() is far faster than repeated concatenation
import time

parts = [str(number) for number in range(20000)]


def with_plus():
    result = ""
    for part in parts:
        result += part
    return result


def with_join():
    return "".join(parts)


start = time.perf_counter()
with_plus()
plus_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
with_join()
join_ms = (time.perf_counter() - start) * 1000

print(f"repeated +=: {plus_ms:7.1f} ms")
print(f"join:        {join_ms:7.1f} ms")

In [ ]:
# EXAMPLE 34: partition is faster than split for one separator
import time

line = "key=" + "x" * 1000
repetitions = 100000


def time_it(operation):
    start = time.perf_counter()
    for _ in range(repetitions):
        operation()
    return (time.perf_counter() - start) * 1000


split_ms = time_it(lambda: line.split("=", 1))
partition_ms = time_it(lambda: line.partition("="))

print(f"split('=', 1): {split_ms:6.1f} ms")
print(f"partition('='): {partition_ms:6.1f} ms")
print("")
print("partition also guarantees three values, so unpacking never fails.")

In [ ]:
# EXAMPLE 35: Why you should not parse real CSV by hand
# split(',') breaks when a field contains a comma inside quotes.
line = 'Asha,"Nikumbh, A.",Mumbai'

print("naive split:", line.split(","))
print("")

import csv
import io

parsed = next(csv.reader(io.StringIO(line)))
print("csv module: ", parsed)
print("")
print("Use the csv module for real data. Chapter 22 covers it.")

In [ ]:
# EXAMPLE 36: A safe key-value parser
def parse_settings(text):
    """Parse lines of key=value into a dictionary."""
    settings = {}

    for line in text.splitlines():
        # Skip blanks and comments.
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        # partition never raises and never returns the wrong shape.
        key, separator, value = stripped.partition("=")

        if separator:
            settings[key.strip()] = value.strip()

    return settings


config = """
# Application settings
timeout = 30
retries = 3

message = hello = world
malformed line
"""

for key, value in parse_settings(config).items():
    print(f"{key:<10} {value!r}")

In [ ]:
# EXAMPLE 37: Splitting on multiple different separators
# String methods split on ONE separator. For several, use re.
import re

text = "a,b;c|d"

print("split(','):", text.split(","))
print("re.split:  ", re.split(r"[,;|]", text))
print("")
print("Chapter 30 covers regular expressions.")

In [ ]:
# EXAMPLE 38: A complete split/join reference
text = "a,b,,c,"

print("text =", repr(text))
print("")

operations = [
    ("text.split(',')", text.split(",")),
    ("text.split(',', 1)", text.split(",", 1)),
    ("text.rsplit(',', 1)", text.rsplit(",", 1)),
    ("text.partition(',')", text.partition(",")),
    ("text.rpartition(',')", text.rpartition(",")),
    ("'-'.join(text.split(','))", "-".join(text.split(","))),
]

for expression, result in operations:
    print(f"{expression:<28} -> {result}")

## Takeaways

1. **`split()` and `split(" ")` are different.** No argument collapses runs of
   whitespace and ignores leading/trailing space. An explicit separator does
   neither.
2. Use `split()` for **human text**, `split(sep)` for **structured data**.
3. **`maxsplit`** stops after N splits — essential when the value may contain the
   separator: `line.split("=", 1)`.
4. **`rsplit(sep, 1)`** is the idiom for separating the last component of a path
   or filename.
5. **`splitlines()`** handles `\n`, `\r\n` and `\r`; `split("\n")` does not.
   `keepends=True` retains the endings.
6. **`partition()` always returns three parts**, so unpacking never fails — it is
   safer than `split()` for parsing.
7. `join()` is called **on the separator** and accepts **any iterable** of
   strings.
8. Splitting on `""` raises `ValueError`; use `list(text)` for characters.
9. **Do not parse real CSV with `split(",")`** — quoted fields break it. Use the
   `csv` module.

## Try it yourself

1. Split `"  a   b  "` both ways and explain the difference.
2. Parse `"message=hello=world"` keeping the full value.
3. Separate `"a/b/c/file.tar.gz"` into directory, name and full extension.
4. Write a config parser using `partition()` that skips comments and blanks.
5. Show a CSV line that `split(",")` gets wrong.